# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamed-6513/flyrank_ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

We will build the feature vector for a mid-panel month (`2026-03` for the label, `2026-02` for features). We create `is_missing_X` flags for missing data rather than blindly filling with zero, ensuring we don't accidentally teach the model that "not tracked" means "zero traffic".

In [4]:
import os
import pandas as pd
import duckdb

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    hf_token = os.environ.get('HF_TOKEN')

if not hf_token:
    raise ValueError('HF_TOKEN not found.')

con = duckdb.connect()
con.execute('INSTALL httpfs;')
con.execute('LOAD httpfs;')
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

query_features = """
WITH content_base AS (
    SELECT
        content_hash_id as content_id,
        client_hash_id as client_id,
        content_type,
        COALESCE(word_count, 0) as word_count,
        CASE WHEN word_count IS NULL THEN 1 ELSE 0 END as is_missing_word_count,
        COALESCE(search_volume, 0) as search_volume,
        CASE WHEN search_volume IS NULL THEN 1 ELSE 0 END as is_missing_search_volume,
        COALESCE(competition, 0) as competition
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
),
perf_mar AS (
    SELECT
        content_hash_id as content_id,
        SUM(gsc_clicks) as clicks_mar
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    GROUP BY content_hash_id
),
perf_feb AS (
    SELECT
        content_hash_id as content_id,
        SUM(gsc_clicks) as clicks_feb,
        SUM(gsc_impressions) as impressions_feb,
        AVG(gsc_avg_position) as avg_position_feb
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
    GROUP BY content_hash_id
)
SELECT
    c.content_id,
    c.client_id,
    c.content_type,
    c.word_count,
    c.is_missing_word_count,
    c.search_volume,
    c.is_missing_search_volume,
    c.competition,
    COALESCE(pf.clicks_feb, 0) as clicks_prev_month,
    COALESCE(pf.impressions_feb, 0) as impressions_prev_month,
    COALESCE(pf.avg_position_feb, 0) as avg_position_prev_month,
    COALESCE(pm.clicks_mar, 0) as label_clicks_current_month
FROM content_base c
JOIN perf_feb pf ON c.content_id = pf.content_id
JOIN perf_mar pm ON c.content_id = pm.content_id
LIMIT 5
"""
df_features = con.execute(query_features).df()
display(df_features)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_id,client_id,content_type,word_count,is_missing_word_count,search_volume,is_missing_search_volume,competition,clicks_prev_month,impressions_prev_month,avg_position_prev_month,label_clicks_current_month
0,content_1eea820697c3b95a,client_e547b89c05043229,keyword article,2613,0,140,0,0.07,0.0,299.0,12.946228,0.0
1,content_9abd8b303f805847,client_e547b89c05043229,keyword article,2992,0,90,0,0.00,6.0,733.0,6.495085,4.0
2,content_5f58c55cbfee172a,client_e547b89c05043229,keyword article,2225,0,720,0,0.00,0.0,514.0,10.490023,0.0
3,content_6fe390ba3af1e456,client_e547b89c05043229,keyword article,2797,0,110,0,0.00,3.0,2931.0,38.436254,5.0
4,content_3ad5d2160242b9ca,client_e547b89c05043229,keyword article,2396,0,70,0,0.00,2.0,970.0,9.710810,1.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

- `word_count`: Number of words. Missing values (e.g. for videos/feeds) are filled with 0, but an `is_missing_word_count` flag is added so the model can learn the difference. Known before prediction.
- `search_volume` & `competition`: Keyword volume/difficulty. Missing values filled with 0, and `is_missing_search_volume` flag added to distinguish true 0 from "not applicable". Known before prediction.
- `clicks_prev_month`, `impressions_prev_month`, `avg_position_prev_month`: Historical traffic data strictly from the month *before* our prediction target. Available before prediction.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Leakage checklist execution:
1. **Label-derived columns**: We are predicting `label_clicks_current_month` (March). Our features (`clicks_prev_month`, `impressions_prev_month`) are strictly from February.
2. **Product flags**: We deliberately avoided including any FlyRank internal scores (like `health_score`, `needs_ctr_fix`) that might encode the answer.
3. **Future overlapping windows**: Our feature aggregation window (Feb 2026) is strictly disjoint from our label window (Mar 2026).

In [6]:
# Verify that label (March) is not leaked into features (February).
leakage_check_query = """
SELECT
    MIN(report_date) as min_feat_date,
    MAX(report_date) as max_feat_date
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
"""
print("Feature Window:")
display(con.execute(leakage_check_query).df())

leakage_check_query_label = """
SELECT
    MIN(report_date) as min_label_date,
    MAX(report_date) as max_label_date
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
print("Label Window:")
display(con.execute(leakage_check_query_label).df())

Feature Window:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_feat_date,max_feat_date
0,2026-02-01,2026-02-28


Label Window:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_label_date,max_label_date
0,2026-03-01,2026-03-31


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- `trend_direction` / `trend_pct` / `is_declining_label` from the starter dataset: Excluded because they encode the very outcome we want to predict (Label-derived leakage).
- `health_score` / `needs_ctr_fix`: Excluded because they are FlyRank product decision flags. Using them would teach the model the existing product rules rather than the true pattern.
- March data in feature calculations: Excluded because it would overlap with the target label (Future Window Leakage).

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.